In [1]:
!pip install pytorch-tabnet -q
!pip install pytorch-tabular[extra] -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.8/165.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.1/165.1 kB 3.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, Conv1D, MaxPooling1D,
    Flatten, LSTM, Input
)
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from pytorch_tabnet.tab_model import TabNetClassifier

import pandas as pd

# LOAD DATASET

DATA_PATH = '/content/drive/MyDrive/WAF_THESIS/DATASETS/sql_cleaned.csv'


df = pd.read_csv(DATA_PATH)
print("Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())



TARGET = 'class'

X = df.drop(columns=[TARGET])
if 'attack_type' in X.columns:
    X = X.drop(columns=['attack_type'])

y = df[TARGET].values

# Encode target labels to 0/1 integers
le = LabelEncoder()
y = le.fit_transform(y)
n_classes = len(np.unique(y))
print(f"\nClasses: {le.classes_}  |  Encoded: {np.unique(y)}")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train / test split  (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape}  |  Test: {X_test.shape}")

# Reshape for CNN / LSTM  (samples, timesteps, features=1)
X_train_dl = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_dl  = X_test.reshape(X_test.shape[0],  X_test.shape[1],  1)

# RESULTS STORAGE

results = []

AVERAGE = 'binary'

def evaluate_model(model_name, y_true, y_pred):
    """Compute and store metrics; print summary."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=AVERAGE, zero_division=0)
    rec  = recall_score(y_true, y_pred, average=AVERAGE, zero_division=0)
    f1   = f1_score(y_true, y_pred, average=AVERAGE, zero_division=0)

    results.append({
        'Model':     model_name,
        'Accuracy':  round(acc,  4),
        'Precision': round(prec, 4),
        'Recall':    round(rec,  4),
        'F1-Score':  round(f1,   4),
    })

    print(f"\n{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1-Score : {f1:.4f}")

# Early-stopping callback (shared)
early_stop = EarlyStopping(monitor='val_loss', patience=3,
                           restore_best_weights=True, verbose=0)

# 1. DNN (MLP)

print("\n[1/5] Training DNN...")
try:
    dnn = Sequential([
        Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
        Dropout(0.3),
        Dense(64,  activation='relu'),
        Dropout(0.2),
        Dense(32,  activation='relu'),
        Dense(1,   activation='sigmoid'),
    ])
    dnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    dnn.fit(X_train, y_train,
            epochs=20, batch_size=64,
            validation_split=0.2,
            callbacks=[early_stop],
            verbose=1)

    pred_dnn = (dnn.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
    evaluate_model("DNN", y_test, pred_dnn)

except Exception as e:
    print(f"  DNN ERROR: {e}")


# 2. CNN

print("\n[2/5] Training CNN...")
try:
    cnn = Sequential([
        Conv1D(64, kernel_size=2, activation='relu',
               input_shape=(X_train.shape[1], 1)),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid'),
    ])
    cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    cnn.fit(X_train_dl, y_train,
            epochs=20, batch_size=64,
            validation_split=0.2,
            callbacks=[early_stop],
            verbose=1)

    pred_cnn = (cnn.predict(X_test_dl, verbose=0) > 0.5).astype(int).flatten()
    evaluate_model("CNN", y_test, pred_cnn)

except Exception as e:
    print(f"  CNN ERROR: {e}")


# 3. LSTM

print("\n[3/5] Training LSTM...")
try:
    lstm_model = Sequential([
        LSTM(64, input_shape=(X_train.shape[1], 1)),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1,  activation='sigmoid'),
    ])
    lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    lstm_model.fit(X_train_dl, y_train,
                   epochs=20, batch_size=64,
                   validation_split=0.2,
                   callbacks=[early_stop],
                   verbose=1)

    pred_lstm = (lstm_model.predict(X_test_dl, verbose=0) > 0.5).astype(int).flatten()
    evaluate_model("LSTM", y_test, pred_lstm)

except Exception as e:
    print(f"  LSTM ERROR: {e}")


# 4. AUTOENCODER

print("\n[4/5] Training Autoencoder...")
try:
    input_dim   = X_train.shape[1]
    inp         = Input(shape=(input_dim,))
    enc         = Dense(64, activation='relu')(inp)
    enc         = Dense(32, activation='relu')(enc)
    dec         = Dense(64, activation='relu')(enc)
    dec         = Dense(input_dim, activation='linear')(dec)
    autoencoder = Model(inputs=inp, outputs=dec)
    autoencoder.compile(optimizer='adam', loss='mse')

    autoencoder.fit(X_train, X_train,
                    epochs=20, batch_size=64,
                    validation_split=0.2,
                    callbacks=[early_stop],
                    verbose=1)

    # Threshold: 95th percentile of TRAINING reconstruction error
    train_recon = autoencoder.predict(X_train, verbose=0)
    train_mse   = np.mean(np.power(X_train - train_recon, 2), axis=1)
    threshold   = np.percentile(train_mse, 95)

    test_recon  = autoencoder.predict(X_test, verbose=0)
    test_mse    = np.mean(np.power(X_test - test_recon, 2), axis=1)
    pred_ae     = (test_mse > threshold).astype(int)

    evaluate_model("Autoencoder", y_test, pred_ae)

except Exception as e:
    print(f"  Autoencoder ERROR: {e}")


# 5. TABNET

print("\n[5/5] Training TabNet...")
try:
    X_train_tab = X_train.astype(np.float32)
    X_test_tab  = X_test.astype(np.float32)
    y_train_tab = y_train.astype(int)
    y_test_tab  = y_test.astype(int)

    tabnet = TabNetClassifier(
        n_d=32, n_a=32,
        n_steps=5,
        gamma=1.5,
        n_independent=2,
        n_shared=2,
        seed=42,
        verbose=1,
    )
    tabnet.fit(
        X_train=X_train_tab,
        y_train=y_train_tab,
        eval_set=[(X_test_tab, y_test_tab)],
        eval_name=['val'],
        eval_metric=['accuracy'],
        max_epochs=50,
        patience=10,
        batch_size=1024,
        virtual_batch_size=256,
        num_workers=0,
        drop_last=False,
    )
    pred_tab = tabnet.predict(X_test_tab)
    evaluate_model("TabNet", y_test, pred_tab)

except ImportError:
    print("  pytorch-tabnet not installed. Skipping TabNet.")
    print("  Run:  !pip install pytorch-tabnet -q")
except Exception as e:
    print(f"  TabNet ERROR: {e}")


# FINAL COMPARATIVE TABLE

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='F1-Score', ascending=False).reset_index(drop=True)

print("\n" + "="*65)
print("  FINAL COMPARATIVE ANALYSIS — Deep Learning Models")
print("="*65)
print(results_df.to_string(index=False))

best = results_df.iloc[0]
print("\n" + "="*65)
print(f"  BEST MODEL: {best['Model']}  |  F1={best['F1-Score']}  |  Acc={best['Accuracy']}")
print("="*65)

# Save results
results_df.to_csv('deep_learning_comparison_result_sql.csv', index=False)
print("\nResults saved → deep_learning_comparison_results.csv")

Dataset Shape: (13477, 9)
Columns: ['stars', 'dashes', 'or_sign', 'and_sign', 'sub_line', 'comment_sign', 'at_sign', 'badwords', 'class']

Classes: [0 1]  |  Encoded: [0 1]
Class distribution: {np.int64(0): np.int64(10190), np.int64(1): np.int64(3287)}

Train: (10781, 8)  |  Test: (2696, 8)

[1/5] Training DNN...
Epoch 1/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.8784 - loss: 0.3108 - val_accuracy: 0.9940 - val_loss: 0.0739
Epoch 2/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9875 - loss: 0.0563 - val_accuracy: 0.9949 - val_loss: 0.0319
Epoch 3/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9925 - loss: 0.0357 - val_accuracy: 0.9949 - val_loss: 0.0302
Epoch 4/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9934 - loss: 0.0296 - val_accuracy: 0.9949 - val_loss: 0.0280
Epoch 5/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9935 - loss: 0.0297 - val_accuracy: 0.9949 - val_loss: 0.0274
Epoch 6/20
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 4m

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, Conv1D, MaxPooling1D,
    Flatten, LSTM, Input
)
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from pytorch_tabnet.tab_model import TabNetClassifier

import pandas as pd

# LOAD DATASET

DATA_PATH = '/content/drive/MyDrive/WAF_THESIS/DATASETS/osc_cleaned.csv'


df = pd.read_csv(DATA_PATH)
print("Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())



TARGET = 'class'

X = df.drop(columns=[TARGET])
if 'attack_type' in X.columns:
    X = X.drop(columns=['attack_type'])

y = df[TARGET].values

# Encode target labels to 0/1 integers
le = LabelEncoder()
y = le.fit_transform(y)
n_classes = len(np.unique(y))
print(f"\nClasses: {le.classes_}  |  Encoded: {np.unique(y)}")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train / test split  (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape}  |  Test: {X_test.shape}")

# Reshape for CNN / LSTM  (samples, timesteps, features=1)
X_train_dl = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_dl  = X_test.reshape(X_test.shape[0],  X_test.shape[1],  1)

# RESULTS STORAGE

results = []

AVERAGE = 'binary'

def evaluate_model(model_name, y_true, y_pred):
    """Compute and store metrics; print summary."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=AVERAGE, zero_division=0)
    rec  = recall_score(y_true, y_pred, average=AVERAGE, zero_division=0)
    f1   = f1_score(y_true, y_pred, average=AVERAGE, zero_division=0)

    results.append({
        'Model':     model_name,
        'Accuracy':  round(acc,  4),
        'Precision': round(prec, 4),
        'Recall':    round(rec,  4),
        'F1-Score':  round(f1,   4),
    })

    print(f"\n{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1-Score : {f1:.4f}")

# Early-stopping callback (shared)
early_stop = EarlyStopping(monitor='val_loss', patience=3,
                           restore_best_weights=True, verbose=0)

# 1. DNN (MLP)

print("\n[1/5] Training DNN...")
try:
    dnn = Sequential([
        Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
        Dropout(0.3),
        Dense(64,  activation='relu'),
        Dropout(0.2),
        Dense(32,  activation='relu'),
        Dense(1,   activation='sigmoid'),
    ])
    dnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    dnn.fit(X_train, y_train,
            epochs=20, batch_size=64,
            validation_split=0.2,
            callbacks=[early_stop],
            verbose=1)

    pred_dnn = (dnn.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
    evaluate_model("DNN", y_test, pred_dnn)

except Exception as e:
    print(f"  DNN ERROR: {e}")


# 2. CNN

print("\n[2/5] Training CNN...")
try:
    cnn = Sequential([
        Conv1D(64, kernel_size=2, activation='relu',
               input_shape=(X_train.shape[1], 1)),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid'),
    ])
    cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    cnn.fit(X_train_dl, y_train,
            epochs=20, batch_size=64,
            validation_split=0.2,
            callbacks=[early_stop],
            verbose=1)

    pred_cnn = (cnn.predict(X_test_dl, verbose=0) > 0.5).astype(int).flatten()
    evaluate_model("CNN", y_test, pred_cnn)

except Exception as e:
    print(f"  CNN ERROR: {e}")


# 3. LSTM

print("\n[3/5] Training LSTM...")
try:
    lstm_model = Sequential([
        LSTM(64, input_shape=(X_train.shape[1], 1)),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1,  activation='sigmoid'),
    ])
    lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    lstm_model.fit(X_train_dl, y_train,
                   epochs=20, batch_size=64,
                   validation_split=0.2,
                   callbacks=[early_stop],
                   verbose=1)

    pred_lstm = (lstm_model.predict(X_test_dl, verbose=0) > 0.5).astype(int).flatten()
    evaluate_model("LSTM", y_test, pred_lstm)

except Exception as e:
    print(f"  LSTM ERROR: {e}")


# 4. AUTOENCODER

print("\n[4/5] Training Autoencoder...")
try:
    input_dim   = X_train.shape[1]
    inp         = Input(shape=(input_dim,))
    enc         = Dense(64, activation='relu')(inp)
    enc         = Dense(32, activation='relu')(enc)
    dec         = Dense(64, activation='relu')(enc)
    dec         = Dense(input_dim, activation='linear')(dec)
    autoencoder = Model(inputs=inp, outputs=dec)
    autoencoder.compile(optimizer='adam', loss='mse')

    autoencoder.fit(X_train, X_train,
                    epochs=20, batch_size=64,
                    validation_split=0.2,
                    callbacks=[early_stop],
                    verbose=1)

    # Threshold: 95th percentile of TRAINING reconstruction error
    train_recon = autoencoder.predict(X_train, verbose=0)
    train_mse   = np.mean(np.power(X_train - train_recon, 2), axis=1)
    threshold   = np.percentile(train_mse, 95)

    test_recon  = autoencoder.predict(X_test, verbose=0)
    test_mse    = np.mean(np.power(X_test - test_recon, 2), axis=1)
    pred_ae     = (test_mse > threshold).astype(int)

    evaluate_model("Autoencoder", y_test, pred_ae)

except Exception as e:
    print(f"  Autoencoder ERROR: {e}")


# 5. TABNET

print("\n[5/5] Training TabNet...")
try:
    X_train_tab = X_train.astype(np.float32)
    X_test_tab  = X_test.astype(np.float32)
    y_train_tab = y_train.astype(int)
    y_test_tab  = y_test.astype(int)

    tabnet = TabNetClassifier(
        n_d=32, n_a=32,
        n_steps=5,
        gamma=1.5,
        n_independent=2,
        n_shared=2,
        seed=42,
        verbose=1,
    )
    tabnet.fit(
        X_train=X_train_tab,
        y_train=y_train_tab,
        eval_set=[(X_test_tab, y_test_tab)],
        eval_name=['val'],
        eval_metric=['accuracy'],
        max_epochs=50,
        patience=10,
        batch_size=1024,
        virtual_batch_size=256,
        num_workers=0,
        drop_last=False,
    )
    pred_tab = tabnet.predict(X_test_tab)
    evaluate_model("TabNet", y_test, pred_tab)

except ImportError:
    print("  pytorch-tabnet not installed. Skipping TabNet.")
    print("  Run:  !pip install pytorch-tabnet -q")
except Exception as e:
    print(f"  TabNet ERROR: {e}")


# FINAL COMPARATIVE TABLE

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='F1-Score', ascending=False).reset_index(drop=True)

print("\n" + "="*65)
print("  FINAL COMPARATIVE ANALYSIS — Deep Learning Models")
print("="*65)
print(results_df.to_string(index=False))

best = results_df.iloc[0]
print("\n" + "="*65)
print(f"  BEST MODEL: {best['Model']}  |  F1={best['F1-Score']}  |  Acc={best['Accuracy']}")
print("="*65)

# Save results
results_df.to_csv('deep_learning_comparison_result_sql.csv', index=False)
print("\nResults saved → deep_learning_comparison_results.csv")

Dataset Shape: (19095, 9)
Columns: ['dashes', 'and_sign', 'dolar_sign', 'or_sign', 'lessthan', 'greaterthan', 'exclamation', 'badwords', 'class']

Classes: [0 1]  |  Encoded: [0 1]
Class distribution: {np.int64(0): np.int64(8416), np.int64(1): np.int64(10679)}

Train: (15276, 8)  |  Test: (3819, 8)

[1/5] Training DNN...
Epoch 1/20
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9598 - loss: 0.1631 - val_accuracy: 0.9794 - val_loss: 0.0624
Epoch 2/20
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9832 - loss: 0.0581 - val_accuracy: 0.9800 - val_loss: 0.0590
Epoch 3/20
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9849 - loss: 0.0502 - val_accuracy: 0.9800 - val_loss: 0.0575
Epoch 4/20
191/191 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9854 - loss: 0.0509 - val_accuracy: 0.9800 - val_loss: 0.0570
Epoch 5/20
191/191 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9851 - loss: 0.0498 - val_accuracy: 0.9800 - val_loss: 0.0559
Epoch 6/20
191/191 ━━━━━━━━━━━━━━━━━

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, Conv1D, MaxPooling1D,
    Flatten, LSTM, Input
)
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from pytorch_tabnet.tab_model import TabNetClassifier

import pandas as pd

# LOAD DATASET

DATA_PATH = '/content/drive/MyDrive/WAF_THESIS/DATASETS/xss_cleaned.csv'


df = pd.read_csv(DATA_PATH)
print("Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())



TARGET = 'class'

X = df.drop(columns=[TARGET])
if 'attack_type' in X.columns:
    X = X.drop(columns=['attack_type'])

y = df[TARGET].values

# Encode target labels to 0/1 integers
le = LabelEncoder()
y = le.fit_transform(y)
n_classes = len(np.unique(y))
print(f"\nClasses: {le.classes_}  |  Encoded: {np.unique(y)}")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train / test split  (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape}  |  Test: {X_test.shape}")

# Reshape for CNN / LSTM  (samples, timesteps, features=1)
X_train_dl = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_dl  = X_test.reshape(X_test.shape[0],  X_test.shape[1],  1)

# RESULTS STORAGE

results = []

AVERAGE = 'binary'

def evaluate_model(model_name, y_true, y_pred):
    """Compute and store metrics; print summary."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=AVERAGE, zero_division=0)
    rec  = recall_score(y_true, y_pred, average=AVERAGE, zero_division=0)
    f1   = f1_score(y_true, y_pred, average=AVERAGE, zero_division=0)

    results.append({
        'Model':     model_name,
        'Accuracy':  round(acc,  4),
        'Precision': round(prec, 4),
        'Recall':    round(rec,  4),
        'F1-Score':  round(f1,   4),
    })

    print(f"\n{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1-Score : {f1:.4f}")

# Early-stopping callback (shared)
early_stop = EarlyStopping(monitor='val_loss', patience=3,
                           restore_best_weights=True, verbose=0)

# 1. DNN (MLP)

print("\n[1/5] Training DNN...")
try:
    dnn = Sequential([
        Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
        Dropout(0.3),
        Dense(64,  activation='relu'),
        Dropout(0.2),
        Dense(32,  activation='relu'),
        Dense(1,   activation='sigmoid'),
    ])
    dnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    dnn.fit(X_train, y_train,
            epochs=20, batch_size=64,
            validation_split=0.2,
            callbacks=[early_stop],
            verbose=1)

    pred_dnn = (dnn.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
    evaluate_model("DNN", y_test, pred_dnn)

except Exception as e:
    print(f"  DNN ERROR: {e}")


# 2. CNN

print("\n[2/5] Training CNN...")
try:
    cnn = Sequential([
        Conv1D(64, kernel_size=2, activation='relu',
               input_shape=(X_train.shape[1], 1)),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid'),
    ])
    cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    cnn.fit(X_train_dl, y_train,
            epochs=20, batch_size=64,
            validation_split=0.2,
            callbacks=[early_stop],
            verbose=1)

    pred_cnn = (cnn.predict(X_test_dl, verbose=0) > 0.5).astype(int).flatten()
    evaluate_model("CNN", y_test, pred_cnn)

except Exception as e:
    print(f"  CNN ERROR: {e}")


# 3. LSTM

print("\n[3/5] Training LSTM...")
try:
    lstm_model = Sequential([
        LSTM(64, input_shape=(X_train.shape[1], 1)),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1,  activation='sigmoid'),
    ])
    lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    lstm_model.fit(X_train_dl, y_train,
                   epochs=20, batch_size=64,
                   validation_split=0.2,
                   callbacks=[early_stop],
                   verbose=1)

    pred_lstm = (lstm_model.predict(X_test_dl, verbose=0) > 0.5).astype(int).flatten()
    evaluate_model("LSTM", y_test, pred_lstm)

except Exception as e:
    print(f"  LSTM ERROR: {e}")


# 4. AUTOENCODER

print("\n[4/5] Training Autoencoder...")
try:
    input_dim   = X_train.shape[1]
    inp         = Input(shape=(input_dim,))
    enc         = Dense(64, activation='relu')(inp)
    enc         = Dense(32, activation='relu')(enc)
    dec         = Dense(64, activation='relu')(enc)
    dec         = Dense(input_dim, activation='linear')(dec)
    autoencoder = Model(inputs=inp, outputs=dec)
    autoencoder.compile(optimizer='adam', loss='mse')

    autoencoder.fit(X_train, X_train,
                    epochs=20, batch_size=64,
                    validation_split=0.2,
                    callbacks=[early_stop],
                    verbose=1)

    # Threshold: 95th percentile of TRAINING reconstruction error
    train_recon = autoencoder.predict(X_train, verbose=0)
    train_mse   = np.mean(np.power(X_train - train_recon, 2), axis=1)
    threshold   = np.percentile(train_mse, 95)

    test_recon  = autoencoder.predict(X_test, verbose=0)
    test_mse    = np.mean(np.power(X_test - test_recon, 2), axis=1)
    pred_ae     = (test_mse > threshold).astype(int)

    evaluate_model("Autoencoder", y_test, pred_ae)

except Exception as e:
    print(f"  Autoencoder ERROR: {e}")


# 5. TABNET

print("\n[5/5] Training TabNet...")
try:
    X_train_tab = X_train.astype(np.float32)
    X_test_tab  = X_test.astype(np.float32)
    y_train_tab = y_train.astype(int)
    y_test_tab  = y_test.astype(int)

    tabnet = TabNetClassifier(
        n_d=32, n_a=32,
        n_steps=5,
        gamma=1.5,
        n_independent=2,
        n_shared=2,
        seed=42,
        verbose=1,
    )
    tabnet.fit(
        X_train=X_train_tab,
        y_train=y_train_tab,
        eval_set=[(X_test_tab, y_test_tab)],
        eval_name=['val'],
        eval_metric=['accuracy'],
        max_epochs=50,
        patience=10,
        batch_size=1024,
        virtual_batch_size=256,
        num_workers=0,
        drop_last=False,
    )
    pred_tab = tabnet.predict(X_test_tab)
    evaluate_model("TabNet", y_test, pred_tab)

except ImportError:
    print("  pytorch-tabnet not installed. Skipping TabNet.")
    print("  Run:  !pip install pytorch-tabnet -q")
except Exception as e:
    print(f"  TabNet ERROR: {e}")


# FINAL COMPARATIVE TABLE

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='F1-Score', ascending=False).reset_index(drop=True)

print("\n" + "="*65)
print("  FINAL COMPARATIVE ANALYSIS — Deep Learning Models")
print("="*65)
print(results_df.to_string(index=False))

best = results_df.iloc[0]
print("\n" + "="*65)
print(f"  BEST MODEL: {best['Model']}  |  F1={best['F1-Score']}  |  Acc={best['Accuracy']}")
print("="*65)

# Save results
results_df.to_csv('deep_learning_comparison_result_sql.csv', index=False)
print("\nResults saved → deep_learning_comparison_results.csv")

Dataset Shape: (30461, 7)
Columns: ['greaterthan', 'lessthan', 'dashes', 'hashes', 'stars', 'badwords', 'class']

Classes: [0 1]  |  Encoded: [0 1]
Class distribution: {np.int64(0): np.int64(10691), np.int64(1): np.int64(19770)}

Train: (24368, 6)  |  Test: (6093, 6)

[1/5] Training DNN...
Epoch 1/20
305/305 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9860 - loss: 0.1060 - val_accuracy: 0.9965 - val_loss: 0.0375
Epoch 2/20
305/305 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9952 - loss: 0.0357 - val_accuracy: 0.9965 - val_loss: 0.0319
Epoch 3/20
305/305 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9954 - loss: 0.0334 - val_accuracy: 0.9965 - val_loss: 0.0278
Epoch 4/20
305/305 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9954 - loss: 0.0303 - val_accuracy: 0.9965 - val_loss: 0.0217
Epoch 5/20
305/305 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9955 - loss: 0.0263 - val_accuracy: 0.9965 - val_loss: 0.0218
Epoch 6/20
305/305 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.995

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, Conv1D, MaxPooling1D,
    Flatten, LSTM, Input
)
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from pytorch_tabnet.tab_model import TabNetClassifier

import pandas as pd

# LOAD DATASET

DATA_PATH = '/content/drive/MyDrive/WAF_THESIS/DATASETS/lfi_cleaned.csv'


df = pd.read_csv(DATA_PATH)
print("Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())



TARGET = 'class'

X = df.drop(columns=[TARGET])
if 'attack_type' in X.columns:
    X = X.drop(columns=['attack_type'])

y = df[TARGET].values

# Encode target labels to 0/1 integers
le = LabelEncoder()
y = le.fit_transform(y)
n_classes = len(np.unique(y))
print(f"\nClasses: {le.classes_}  |  Encoded: {np.unique(y)}")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train / test split  (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape}  |  Test: {X_test.shape}")

# Reshape for CNN / LSTM  (samples, timesteps, features=1)
X_train_dl = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_dl  = X_test.reshape(X_test.shape[0],  X_test.shape[1],  1)

# RESULTS STORAGE

results = []

AVERAGE = 'binary'

def evaluate_model(model_name, y_true, y_pred):
    """Compute and store metrics; print summary."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=AVERAGE, zero_division=0)
    rec  = recall_score(y_true, y_pred, average=AVERAGE, zero_division=0)
    f1   = f1_score(y_true, y_pred, average=AVERAGE, zero_division=0)

    results.append({
        'Model':     model_name,
        'Accuracy':  round(acc,  4),
        'Precision': round(prec, 4),
        'Recall':    round(rec,  4),
        'F1-Score':  round(f1,   4),
    })

    print(f"\n{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1-Score : {f1:.4f}")

# Early-stopping callback (shared)
early_stop = EarlyStopping(monitor='val_loss', patience=3,
                           restore_best_weights=True, verbose=0)

# 1. DNN (MLP)

print("\n[1/5] Training DNN...")
try:
    dnn = Sequential([
        Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
        Dropout(0.3),
        Dense(64,  activation='relu'),
        Dropout(0.2),
        Dense(32,  activation='relu'),
        Dense(1,   activation='sigmoid'),
    ])
    dnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    dnn.fit(X_train, y_train,
            epochs=20, batch_size=64,
            validation_split=0.2,
            callbacks=[early_stop],
            verbose=1)

    pred_dnn = (dnn.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
    evaluate_model("DNN", y_test, pred_dnn)

except Exception as e:
    print(f"  DNN ERROR: {e}")


# 2. CNN

print("\n[2/5] Training CNN...")
try:
    cnn = Sequential([
        Conv1D(64, kernel_size=2, activation='relu',
               input_shape=(X_train.shape[1], 1)),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid'),
    ])
    cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    cnn.fit(X_train_dl, y_train,
            epochs=20, batch_size=64,
            validation_split=0.2,
            callbacks=[early_stop],
            verbose=1)

    pred_cnn = (cnn.predict(X_test_dl, verbose=0) > 0.5).astype(int).flatten()
    evaluate_model("CNN", y_test, pred_cnn)

except Exception as e:
    print(f"  CNN ERROR: {e}")


# 3. LSTM

print("\n[3/5] Training LSTM...")
try:
    lstm_model = Sequential([
        LSTM(64, input_shape=(X_train.shape[1], 1)),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1,  activation='sigmoid'),
    ])
    lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    lstm_model.fit(X_train_dl, y_train,
                   epochs=20, batch_size=64,
                   validation_split=0.2,
                   callbacks=[early_stop],
                   verbose=1)

    pred_lstm = (lstm_model.predict(X_test_dl, verbose=0) > 0.5).astype(int).flatten()
    evaluate_model("LSTM", y_test, pred_lstm)

except Exception as e:
    print(f"  LSTM ERROR: {e}")


# 4. AUTOENCODER

print("\n[4/5] Training Autoencoder...")
try:
    input_dim   = X_train.shape[1]
    inp         = Input(shape=(input_dim,))
    enc         = Dense(64, activation='relu')(inp)
    enc         = Dense(32, activation='relu')(enc)
    dec         = Dense(64, activation='relu')(enc)
    dec         = Dense(input_dim, activation='linear')(dec)
    autoencoder = Model(inputs=inp, outputs=dec)
    autoencoder.compile(optimizer='adam', loss='mse')

    autoencoder.fit(X_train, X_train,
                    epochs=20, batch_size=64,
                    validation_split=0.2,
                    callbacks=[early_stop],
                    verbose=1)

    # Threshold: 95th percentile of TRAINING reconstruction error
    train_recon = autoencoder.predict(X_train, verbose=0)
    train_mse   = np.mean(np.power(X_train - train_recon, 2), axis=1)
    threshold   = np.percentile(train_mse, 95)

    test_recon  = autoencoder.predict(X_test, verbose=0)
    test_mse    = np.mean(np.power(X_test - test_recon, 2), axis=1)
    pred_ae     = (test_mse > threshold).astype(int)

    evaluate_model("Autoencoder", y_test, pred_ae)

except Exception as e:
    print(f"  Autoencoder ERROR: {e}")


# 5. TABNET

print("\n[5/5] Training TabNet...")
try:
    X_train_tab = X_train.astype(np.float32)
    X_test_tab  = X_test.astype(np.float32)
    y_train_tab = y_train.astype(int)
    y_test_tab  = y_test.astype(int)

    tabnet = TabNetClassifier(
        n_d=32, n_a=32,
        n_steps=5,
        gamma=1.5,
        n_independent=2,
        n_shared=2,
        seed=42,
        verbose=1,
    )
    tabnet.fit(
        X_train=X_train_tab,
        y_train=y_train_tab,
        eval_set=[(X_test_tab, y_test_tab)],
        eval_name=['val'],
        eval_metric=['accuracy'],
        max_epochs=50,
        patience=10,
        batch_size=1024,
        virtual_batch_size=256,
        num_workers=0,
        drop_last=False,
    )
    pred_tab = tabnet.predict(X_test_tab)
    evaluate_model("TabNet", y_test, pred_tab)

except ImportError:
    print("  pytorch-tabnet not installed. Skipping TabNet.")
    print("  Run:  !pip install pytorch-tabnet -q")
except Exception as e:
    print(f"  TabNet ERROR: {e}")


# FINAL COMPARATIVE TABLE

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='F1-Score', ascending=False).reset_index(drop=True)

print("\n" + "="*65)
print("  FINAL COMPARATIVE ANALYSIS — Deep Learning Models")
print("="*65)
print(results_df.to_string(index=False))

best = results_df.iloc[0]
print("\n" + "="*65)
print(f"  BEST MODEL: {best['Model']}  |  F1={best['F1-Score']}  |  Acc={best['Accuracy']}")
print("="*65)

# Save results
results_df.to_csv('deep_learning_comparison_result_sql.csv', index=False)
print("\nResults saved → deep_learning_comparison_results.csv")

Dataset Shape: (31177, 8)
Columns: ['sign1', 'sign2', 'sign3', 'sign4', 'sign5', 'sign6', 'badwords', 'class']

Classes: [0 1]  |  Encoded: [0 1]
Class distribution: {np.int64(0): np.int64(8461), np.int64(1): np.int64(22716)}

Train: (24941, 7)  |  Test: (6236, 7)

[1/5] Training DNN...
Epoch 1/20
312/312 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9314 - loss: 0.2467 - val_accuracy: 0.9365 - val_loss: 0.1994
Epoch 2/20
312/312 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9387 - loss: 0.2037 - val_accuracy: 0.9371 - val_loss: 0.1988
Epoch 3/20
312/312 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9388 - loss: 0.2021 - val_accuracy: 0.9373 - val_loss: 0.1982
Epoch 4/20
312/312 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9389 - loss: 0.2009 - val_accuracy: 0.9373 - val_loss: 0.1975
Epoch 5/20
312/312 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9389 - loss: 0.1999 - val_accuracy: 0.9375 - val_loss: 0.1994
Epoch 6/20
312/312 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9390 -